In [1]:
import random

In [2]:
def hadamard_ratio(basis):
    n = basis.dimensions()[0]
    if not basis.is_square():
        raise Exception("no non-square matrices allowed")
    d = abs(basis.determinant())
    l = prod([v.norm() for v in basis])
    return RR((d/l)**(1/n))

In [3]:
def random_unimodular_matrix(n, d):
    up = random_matrix(ZZ, n, x=-d, y=d+1).list()
    low = random_matrix(ZZ, n, x=-d, y=d+1).list()
    for i in range(n):
        for j in range(n):
            if i < j :
                up[i*n + j] = 0
                continue
            if i == j :
                up[i*n + j] = random.choice([-1, 1])
                low[i*n + j] = random.choice([-1, 1])
                continue
            if i > j :
                low[i*n + j] = 0
                continue
    return Matrix(ZZ, n, n, up)*Matrix(ZZ, n, n, low)

In [4]:
def generate_private(n, d):
    while True:
        priv = d * identity_matrix(n)
        priv[0] = vector(ZZ, [randint(-d, d) for _ in range(n)])
        if priv.is_singular() or hadamard_ratio(priv) < 0.9:
            continue
        return priv

In [5]:
def generate_public(priv):
    n = priv.dimensions()[0]
    public = random_unimodular_matrix(n, 10) * priv
    while hadamard_ratio(public) > 0.01:
        public = random_unimodular_matrix(n, 10) * public
    return public


In [6]:
def get_message(n, d):
    return random_vector(ZZ, n, x=-d, y=d+1)


In [7]:
def encrypt(pub, m, e):
    n = pub.dimensions()[0]
    r = random_vector(ZZ, n, x=-e, y=e+1)
    return (m*pub) + r

In [8]:
def decrypt(c, priv, pub):
    mw = cvp_rounding(c, priv)
    return mw*pub.inverse()

In [9]:
def cvp_rounding(target, basis):
    t = target.change_ring(RR)
    br = basis.change_ring(RR)
    c = vector(ZZ, [round(i) for i in t*br.inverse()])
    return c*basis

In [10]:
def test_correctness(n, d, e):
    a = generate_private(n, d)
    b = generate_public(a)
    m = get_message(n, d)
    c = encrypt(b, m, e)
    return decrypt(c, a, b) == m


In [11]:
def test_security(n, d, e):
    a = generate_private(n, d)
    b = generate_public(a)
    m = get_message(n, d)
    c = encrypt(b, m, e)
    return decrypt(c, b, b) != m
